# 11 — Clean-subset within-session + LOSO (v2)

Sensitivity analyses across notebooks 08, 10 and the within-session run
of v1 of this notebook converged on a small set of structural findings:

- A 4.6 % cell-type ↔ layer-label mismatch concentrated in 4 specific
  scans (notebook 10).
- Within-session decoding correlates inversely with mismatch rate
  (notebook 11 v1).
- Per-session zscoring helps GKF but hurts LOSO (notebook 07 §11d).
- DeepSets raw is the most cross-session-robust model (notebook 07
  §11d): LOSO 0.520, beating Tier G HGB LOSO 0.497 and HGB raw/zscored.

This notebook tests how the picture changes when we restrict to the
*clean* dataset: mismatch-excluded neurons, then balanced sessions
post-cleaning. Two experiments:

**(A) Within-session GKF on one randomly-picked balanced session**
  — establishes the within-session ceiling under the cleanest data.

**(B) LOSO on the pool of well-behaved balanced sessions** — same
  models, rotating each session as held-out.

**Five model-normalization combinations** under each experiment:

| | model | preprocessing |
|---|---|---|
| 1 | HGB | long-row A1+B+C1+D1, raw |
| 2 | HGB | long-row A1+B+C1+D1, zscored (within-session) |
| 3 | DeepSets | long-row A1+B+C1+D1, raw |
| 4 | DeepSets | long-row A1+B+C1+D1, zscored (within-session) |
| 5 | HGB | Tier G, raw |

**Reads.**

- `data/processed/tables/units_working.parquet` — full population.
- `data/processed/features/{A1_*,B_per_hash,C1_per_hash,D_per_hash,G_per_neuron}.parquet`.
- `data/processed/splits/cv_assignments.parquet`.

**Writes.**

- `data/processed/results/clean_subset_per_session.parquet`
- `data/processed/results/clean_within_session_runs.parquet`
- `data/processed/results/clean_loso_runs.parquet`
- `reports/figures/11_clean_*.png`

## 1. Setup

In [ ]:
from __future__ import annotations

import sys, time, gc, warnings
from pathlib import Path

if '..' not in sys.path:
    sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

from src.config import (
    PROCESSED_TABLES_DIR, PROCESSED_FEATURES_DIR, PROCESSED_SPLITS_DIR,
    PROCESSED_RESULTS_DIR, FIGURES_DIR, RANDOM_SEED, ensure_dirs,
)
from src.data.loaders import load_working_pop, build_modeling_table
from src.eval.metrics import neuron_level_score, summarize_cv_runs
from src.features.tier_b import B_FEATURE_NAMES
from src.features.tier_c import C1_FEATURE_NAMES
from src.features.tier_d import D_FEATURE_NAMES
from src.models.deep_sets import DeepSets, train_deep_sets

ensure_dirs()
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

CLASSES = np.array(['L2/3', 'L4', 'L5', 'L6'])
label_to_idx = {c: i for i, c in enumerate(CLASSES)}
N_FOLDS = 5
STIM_OHE = ['d_stim_Clip', 'd_stim_Monet2', 'd_stim_Trippy']
ALL_D_COLS = list(D_FEATURE_NAMES) + STIM_OHE

# Cell-type → layer mapping (validated in notebook 10).
CELL_TYPE_TO_LAYER = {
    '23P':   'L2/3',
    '4P':    'L4',
    '5P-IT': 'L5',
    '5P-ET': 'L5',
    '5P-NP': 'L5',
    '6P-IT': 'L6',
    '6P-CT': 'L6',
}

# Threshold for 'well-behaved' session selection.
MIN_COUNT_PER_CLASS = 20  # post-cleaning min cells per layer class


def make_hgb(random_state: int = RANDOM_SEED):
    return HistGradientBoostingClassifier(
        max_iter=100, max_depth=8, learning_rate=0.05,
        l2_regularization=1.0,
        early_stopping=True, n_iter_no_change=10,
        random_state=random_state,
    )


def per_session_zscore(X: np.ndarray, sessions: np.ndarray) -> np.ndarray:
    """Median + 1.4826·MAD per session per feature column.
    For multi-session data this removes session offsets. For single-session
    (within-session experiment) it degenerates into RobustScaler-style normalization"""
    X = np.asarray(X, dtype=np.float64).copy()
    out = np.full_like(X, np.nan)
    for sk in np.unique(sessions):
        mask = sessions == sk
        Xs = X[mask]
        med = np.nanmedian(Xs, axis=0)
        mad = np.nanmedian(np.abs(Xs - med), axis=0)
        scale = np.where(mad > 1e-9, 1.4826 * mad, 1.0)
        out[mask] = (Xs - med) / scale
    return out

print('cwd:', Path.cwd())
print('MIN_COUNT_PER_CLASS:', MIN_COUNT_PER_CLASS)

## 2. Theory — why this design

Three converging Phase-1 sensitivity findings motivate restricting to
the cleanest possible subset:

1. **Mismatched neurons add label noise**. 4.6 % of working-population
   neurons have cell-type and layer-label disagreeing. Three quarters
   are biologically interpretable boundary cells; ~25 % (the L6 ones)
   have no boundary signature and may be EM cell-type errors. Removing
   the entire 4.6 % is a conservative cleanliness move.

2. **The 4 'session-6' scans are systematically harder** (notebook 10
   + notebook 11 v1). They concentrate the labelling mismatch and have
   the lowest within-session and balanced-LOSO bal_acc.

3. **Per-session zscoring helps GKF but hurts LOSO** (notebook 07
   §11d). For the cross-session story, raw is the honest framing for
   long-row models too.

What this notebook adds: **how do the Phase-1 numbers change when we
remove all three sources of noise simultaneously?**

### Methodological note: zscoring within one session

When we apply per-session zscoring inside *one* session, the function
becomes RobustScaler — it normalises feature scales using the within-
session median and MAD, but does **not** remove cross-session offsets
(there are none). For tree-based HGB, zscoring is essentially neutral
(splits are scale-invariant). For DeepSets' MLP encoder, zscoring may
matter for numerical stability. **For Tier G specifically** (116
features all in `amp_mean` units), within-session zscoring is
essentially a no-op for HGB; we therefore test only the raw variant.

For the long-row stack we test both raw and zscored within-session,
to verify that the §11d 'raw > zscored under LOSO' pattern still holds
in the within-session limit.

## 3. Build the clean dataset

Apply `consistent = (CELL_TYPE_TO_LAYER[celltype_label] == layer_label)`
and drop neurons with `consistent == 0`.

In [ ]:
units = load_working_pop()
units['celltype_layer'] = units['celltype_label'].map(CELL_TYPE_TO_LAYER)
assert units['celltype_layer'].notna().all()
units['consistent'] = (units['celltype_layer'] == units['layer_label']).astype(int)

n_full = len(units)
n_consistent = (units['consistent']==1).sum()
n_mismatch = n_full - n_consistent
print(f'Full population : {n_full:,}')
print(f'Consistent      : {n_consistent:,}')
print(f'Mismatched      : {n_mismatch:,}  ({n_mismatch/n_full*100:.2f}%)')

units_clean = units[units['consistent']==1].copy()
consistent_nuclei = set(units_clean['nucleus_id'])
print(f'Clean working population: {len(units_clean):,}')

## 4. Per-session composition (post-cleaning) + well-behaved selection

Compute per-session class counts after mismatch removal. A session is
**well-behaved** if it has all four layers represented with at least
`MIN_COUNT_PER_CLASS = 20` neurons each.

Reading rule: small per-class counts hurt class-balanced training and
blow up per-class recall variance. The 20-cell threshold is a practical
minimum for HGB / DeepSets to learn the class without falling back to
the prior at every fold.

In [ ]:
PER_SESS_PATH = PROCESSED_RESULTS_DIR / 'clean_subset_per_session.parquet'

rows = []
for sk, grp in units_clean.groupby('session_key'):
    counts = grp['layer_label'].value_counts()
    n_classes = int((counts > 0).sum())
    min_count = int(counts.reindex(['L2/3','L4','L5','L6'], fill_value=0).min())
    rows.append({
        'session_key': sk,
        'n_neurons': len(grp),
        'n_L2_3': int(counts.get('L2/3', 0)),
        'n_L4':   int(counts.get('L4', 0)),
        'n_L5':   int(counts.get('L5', 0)),
        'n_L6':   int(counts.get('L6', 0)),
        'n_classes_present': n_classes,
        'min_count_per_class': min_count,
        'well_behaved': bool(n_classes == 4 and min_count >= MIN_COUNT_PER_CLASS),
    })
df_sess = pd.DataFrame(rows).sort_values('session_key')
df_sess.to_parquet(PER_SESS_PATH)

print('=== Per-session composition AFTER mismatch removal ===')
print(df_sess.to_string(index=False))
print()
well_behaved = df_sess[df_sess['well_behaved']].copy()
WELL_BEHAVED_SESSIONS = list(well_behaved['session_key'])
print(f'Well-behaved sessions ({len(WELL_BEHAVED_SESSIONS)}): {WELL_BEHAVED_SESSIONS}')
print(f'Total neurons in well-behaved pool: '
      f'{units_clean[units_clean["session_key"].isin(WELL_BEHAVED_SESSIONS)].shape[0]:,}')

## 5. Random pick — one session for the within-session experiment

Reproducible random pick (seed = 42) of one session from the
well-behaved pool. The chosen session will run all 5 model-normalisation
combinations under within-session GroupKFold(`nucleus_id`).

In [ ]:
rng = np.random.default_rng(42)
WITHIN_SESSION = str(rng.choice(WELL_BEHAVED_SESSIONS))
print(f'Random session for within-session experiment: {WITHIN_SESSION}')
n_within = (units_clean['session_key']==WITHIN_SESSION).sum()
print(f'Neurons in {WITHIN_SESSION}: {n_within:,}')
print(f'Class composition (clean, in {WITHIN_SESSION}):')
print(units_clean[units_clean['session_key']==WITHIN_SESSION]['layer_label']
      .value_counts().reindex(['L2/3','L4','L5','L6'], fill_value=0).to_string())

## 6. Build feature matrices for the well-behaved pool

We keep two parallel matrices: the long-row A1+B+C1+D1 table (1 row
per (neuron, hash)) and the per-neuron Tier G fingerprint. Both are
filtered to consistent neurons in well-behaved sessions.

In [ ]:
# Long-row A1+B+C1+D1 — build via the canonical loader, then filter.
X_a1, y_a1, g_a1, f_a1, df_a1 = build_modeling_table(
    level='A1', blocks=['amp', 'shape'])
feat_a1 = [c for c in df_a1.columns if c.startswith(('amp_', 'shape_'))]

# Merge B / C1 / D.
b_hash = pd.read_parquet(
    PROCESSED_FEATURES_DIR / 'B_per_hash.parquet',
    columns=['nucleus_id', 'condition_hash'] + list(B_FEATURE_NAMES))
df_full = df_a1.merge(b_hash, on=['nucleus_id', 'condition_hash'],
                      how='inner', validate='one_to_one')
c1 = pd.read_parquet(
    PROCESSED_FEATURES_DIR / 'C1_per_hash.parquet',
    columns=['nucleus_id', 'condition_hash'] + list(C1_FEATURE_NAMES))
df_full = df_full.merge(c1, on=['nucleus_id', 'condition_hash'],
                        how='left', validate='one_to_one')
d_full = pd.read_parquet(PROCESSED_FEATURES_DIR / 'D_per_hash.parquet')
d_for_join = d_full.copy()
for fam in ['Clip', 'Monet2', 'Trippy']:
    d_for_join[f'd_stim_{fam}'] = (d_for_join['stim_type'] == fam).astype(np.float32)
d_for_join = d_for_join[['condition_hash'] + list(D_FEATURE_NAMES) + STIM_OHE]
df_full = df_full.merge(d_for_join, on='condition_hash',
                        how='left', validate='many_to_one')

# Filter to clean neurons in well-behaved sessions.
df_full = df_full[df_full['nucleus_id'].isin(consistent_nuclei)]
df_full = df_full[df_full['session_key'].isin(WELL_BEHAVED_SESSIONS)].copy()
feat_full = feat_a1 + list(B_FEATURE_NAMES) + list(C1_FEATURE_NAMES) + ALL_D_COLS
print(f'Long-row clean+well-behaved table: {df_full.shape}')

# Tier G — per-neuron, then filter same way.
g_wide = pd.read_parquet(PROCESSED_FEATURES_DIR / 'G_per_neuron.parquet')
g_cols = [c for c in g_wide.columns if c.startswith('g_')]
df_g = (g_wide.merge(units_clean[['nucleus_id','session_key','layer_label']],
                      on='nucleus_id', how='inner', validate='one_to_one'))
df_g = df_g[df_g['session_key'].isin(WELL_BEHAVED_SESSIONS)].copy()
print(f'Tier G clean+well-behaved table: {df_g.shape}')

del X_a1, y_a1, g_a1, f_a1, df_a1, b_hash, c1, d_for_join, d_full, g_wide
gc.collect()

## 7. Helpers — set-tensor builder + GKF / LOSO loops

In [ ]:
def build_set_tensors(X_in, y_in, g_in, mask):
    """Build per-neuron set tensors from a row-level subset."""
    X_sub = X_in[mask]; y_sub = y_in[mask]; g_sub = g_in[mask]
    imp = SimpleImputer(strategy='median')
    X_sub = imp.fit_transform(X_sub).astype(np.float32)
    neuron_ids = np.unique(g_sub)
    by_n = {nid: i for i, nid in enumerate(neuron_ids)}
    counts = pd.Series(g_sub).value_counts()
    max_set = int(counts.max())
    n_features = X_sub.shape[1]
    sets = np.zeros((len(neuron_ids), max_set, n_features), dtype=np.float32)
    set_mask = np.zeros((len(neuron_ids), max_set), dtype=np.float32)
    y_neur = np.zeros(len(neuron_ids), dtype=np.int64)
    order = np.argsort(g_sub, kind='stable')
    g_sorted = g_sub[order]; X_sorted = X_sub[order]; y_sorted = y_sub[order]
    start = 0
    while start < len(g_sorted):
        nid = g_sorted[start]; end = start + 1
        while end < len(g_sorted) and g_sorted[end] == nid:
            end += 1
        i = by_n[nid]
        n_rows = end - start
        sets[i, :n_rows, :] = X_sorted[start:end]
        set_mask[i, :n_rows] = 1.0
        y_neur[i] = label_to_idx[y_sorted[start]]
        start = end
    return sets, set_mask, y_neur, neuron_ids


def deepsets_train_eval(X_data, y, g, train_mask, test_mask, n_features,
                         n_epochs=30, batch_size=128, lr=1e-3,
                         random_state=RANDOM_SEED):
    """Train DeepSets on train_mask rows, evaluate on test_mask rows."""
    sets_tr, mask_tr, y_neur_tr, neur_tr = build_set_tensors(X_data, y, g, train_mask)
    sets_te, mask_te, y_neur_te, neur_te = build_set_tensors(X_data, y, g, test_mask)
    cw_np = compute_sample_weight('balanced', y_neur_tr)
    cw = np.zeros(len(CLASSES), dtype=np.float32)
    for c in range(len(CLASSES)):
        m = y_neur_tr == c
        if m.any():
            cw[c] = cw_np[m].mean()
    Xtr_t = torch.from_numpy(sets_tr); Mtr_t = torch.from_numpy(mask_tr); Ytr_t = torch.from_numpy(y_neur_tr)
    Xte_t = torch.from_numpy(sets_te); Mte_t = torch.from_numpy(mask_te); Yte_t = torch.from_numpy(y_neur_te)
    torch.manual_seed(random_state)
    model = DeepSets(n_features=n_features, n_classes=len(CLASSES),
                     enc_hidden=64, head_hidden=64).to('cpu')
    train_deep_sets(model, Xtr_t, Mtr_t, Ytr_t, Xte_t, Mte_t, Yte_t,
                    class_weights=torch.from_numpy(cw),
                    n_epochs=n_epochs, batch_size=batch_size, lr=lr,
                    verbose=False)
    model.eval()
    with torch.no_grad():
        logits = model(Xte_t, Mte_t)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    sc = neuron_level_score(
        y_row_true=CLASSES[Yte_t.numpy()],
        y_row_proba=probs,
        groups=neur_te,
        classes=CLASSES,
    )
    return sc

## 8. Experiment A — within-session GKF on the random session

5-fold GroupKFold(`nucleus_id`) inside `WITHIN_SESSION`. Five model-
normalization combinations. All cached to
`clean_within_session_runs.parquet`.

In [ ]:
WITHIN_PATH = PROCESSED_RESULTS_DIR / 'clean_within_session_runs.parquet'
rows_within: list[dict] = []
if WITHIN_PATH.exists():
    rows_within = pd.read_parquet(WITHIN_PATH).to_dict(orient='records')
    print(f'Loaded {len(rows_within)} cached within-session rows.')
done_keys = {(r['model'], r['preprocessing'], r['features']) for r in rows_within}

# Filter long-row table to the random session.
df_w = df_full[df_full['session_key'] == WITHIN_SESSION].copy()
X_long = df_w[feat_full].to_numpy(dtype=np.float64)
X_long_zs = per_session_zscore(X_long, df_w['session_key'].to_numpy())
y_long = df_w['layer_label'].to_numpy()
g_long = df_w['nucleus_id'].to_numpy()

# Build within-session 5-fold GroupKFold(nucleus_id).
unique_n = np.unique(g_long)
rng = np.random.default_rng(RANDOM_SEED)
rng.shuffle(unique_n)
fold_for_nuc = {nid: i % N_FOLDS for i, nid in enumerate(unique_n)}
fold_long = np.array([fold_for_nuc[n] for n in g_long], dtype=np.int8)

# Tier G subset for the random session.
df_g_w = df_g[df_g['session_key'] == WITHIN_SESSION].copy()
X_g = df_g_w[g_cols].to_numpy(dtype=np.float64)
y_g = df_g_w['layer_label'].to_numpy()
g_g = df_g_w['nucleus_id'].to_numpy()
fold_g = np.array([fold_for_nuc.get(n, 0) for n in g_g], dtype=np.int8)

print(f'Within-session table for {WITHIN_SESSION}:')
print(f'  long-row: {df_w.shape};  Tier G: {df_g_w.shape}')

configs = [
    ('HGB',      'raw',     'A1+B+C1+D1', X_long,    y_long, g_long, fold_long),
    ('HGB',      'zscored', 'A1+B+C1+D1', X_long_zs, y_long, g_long, fold_long),
    ('DeepSets', 'raw',     'A1+B+C1+D1', X_long,    y_long, g_long, fold_long),
    ('DeepSets', 'zscored', 'A1+B+C1+D1', X_long_zs, y_long, g_long, fold_long),
    ('HGB',      'raw',     'TierG',      X_g,       y_g,    g_g,    fold_g),
]

for model_name, prep, feats, X_, y_, g_, f_ in configs:
    if (model_name, prep, feats) in done_keys:
        print(f'  [{model_name}-{prep:<8}-{feats:<10}] SKIP (cached)')
        continue
    t0 = time.time()
    fold_metrics = []
    for k in range(N_FOLDS):
        tr = f_ != k; te = f_ == k
        if not tr.any() or not te.any() or len(np.unique(y_[te])) < 2:
            continue
        if model_name == 'HGB':
            clf = make_hgb()
            sw = compute_sample_weight('balanced', y_[tr])
            clf.fit(X_[tr], y_[tr], sample_weight=sw)
            probs = clf.predict_proba(X_[te])
            sc = neuron_level_score(y_[te], probs, g_[te], clf.classes_)
        else:  # DeepSets
            sc = deepsets_train_eval(X_, y_, g_, tr, te, n_features=X_.shape[1])
        fold_metrics.append(sc)
    if not fold_metrics:
        print(f'  [{model_name}-{prep:<8}-{feats:<10}] FAILED')
        continue
    s = summarize_cv_runs(fold_metrics)
    rows_within.append({
        'model': model_name, 'preprocessing': prep, 'features': feats,
        'session': WITHIN_SESSION,
        'n_neurons': len(np.unique(g_)),
        'balanced_accuracy': s['balanced_accuracy'],
        'balanced_accuracy_std': s['balanced_accuracy_std'],
        'macro_f1': s['macro_f1'], 'macro_f1_std': s['macro_f1_std'],
        **{f'recall_{c}': s['per_class_recall'].get(c) for c in ['L2/3','L4','L5','L6']},
    })
    pd.DataFrame(rows_within).to_parquet(WITHIN_PATH)
    print(f'  [{model_name}-{prep:<8}-{feats:<10}] '
          f'bal_acc={s["balanced_accuracy"]:.3f} ± {s["balanced_accuracy_std"]:.3f}  '
          f'f1={s["macro_f1"]:.3f}  | {time.time()-t0:.1f}s | saved')

df_within = pd.DataFrame(rows_within)
print()
print('=== Within-session GKF on session', WITHIN_SESSION, '===')
print(df_within.round(3).to_string(index=False))

## 9. Experiment B — LOSO on the well-behaved pool

Same 5 model-normalization combinations, but training on (well-behaved
minus held-out) and testing on each held-out scan in turn.

Per-session zscoring is computed across the *training* sessions only;
the held-out scan is zscored using its own statistics (the standard
LOSO contract).

In [ ]:
LOSO_PATH = PROCESSED_RESULTS_DIR / 'clean_loso_runs.parquet'
rows_loso: list[dict] = []
if LOSO_PATH.exists():
    rows_loso = pd.read_parquet(LOSO_PATH).to_dict(orient='records')
    print(f'Loaded {len(rows_loso)} cached LOSO rows.')
done_keys_loso = {(r['model'], r['preprocessing'], r['features'], r['held_out_scan']) for r in rows_loso}

# Long-row arrays (already filtered to clean+well-behaved).
X_long_full = df_full[feat_full].to_numpy(dtype=np.float64)
y_long_full = df_full['layer_label'].to_numpy()
g_long_full = df_full['nucleus_id'].to_numpy()
s_long_full = df_full['session_key'].to_numpy()
X_long_zs_full = per_session_zscore(X_long_full, s_long_full)

# Tier G arrays.
X_g_full = df_g[g_cols].to_numpy(dtype=np.float64)
y_g_full = df_g['layer_label'].to_numpy()
g_g_full = df_g['nucleus_id'].to_numpy()
s_g_full = df_g['session_key'].to_numpy()

configs_loso = [
    ('HGB',      'raw',     'A1+B+C1+D1', X_long_full,    y_long_full, g_long_full, s_long_full),
    ('HGB',      'zscored', 'A1+B+C1+D1', X_long_zs_full, y_long_full, g_long_full, s_long_full),
    ('DeepSets', 'raw',     'A1+B+C1+D1', X_long_full,    y_long_full, g_long_full, s_long_full),
    ('DeepSets', 'zscored', 'A1+B+C1+D1', X_long_zs_full, y_long_full, g_long_full, s_long_full),
    ('HGB',      'raw',     'TierG',      X_g_full,       y_g_full,    g_g_full,    s_g_full),
]

for model_name, prep, feats, X_, y_, g_, s_ in configs_loso:
    print(f'\n--- {model_name} {prep} {feats} ---')
    for held_out in WELL_BEHAVED_SESSIONS:
        if (model_name, prep, feats, held_out) in done_keys_loso:
            continue
        tr = s_ != held_out; te = s_ == held_out
        if not tr.any() or not te.any():
            continue
        t0 = time.time()
        if model_name == 'HGB':
            clf = make_hgb()
            sw = compute_sample_weight('balanced', y_[tr])
            clf.fit(X_[tr], y_[tr], sample_weight=sw)
            probs = clf.predict_proba(X_[te])
            sc = neuron_level_score(y_[te], probs, g_[te], clf.classes_)
        else:
            sc = deepsets_train_eval(X_, y_, g_, tr, te, n_features=X_.shape[1])
        rows_loso.append({
            'model': model_name, 'preprocessing': prep, 'features': feats,
            'held_out_scan': held_out,
            'n_test_neurons': sc['n_neurons'],
            'balanced_accuracy': sc['balanced_accuracy'],
            'macro_f1': sc['macro_f1'],
            **{f'recall_{c}': sc['per_class_recall'].get(c) for c in ['L2/3','L4','L5','L6']},
        })
        pd.DataFrame(rows_loso).to_parquet(LOSO_PATH)
        print(f'  held-out={held_out}: bal_acc={sc["balanced_accuracy"]:.3f}  f1={sc["macro_f1"]:.3f}  | {time.time()-t0:.1f}s')

df_loso = pd.DataFrame(rows_loso)

In [ ]:
# LOSO summary table (mean ± std across well-behaved scans).
summary = (df_loso.groupby(['model','preprocessing','features'])
           .agg(bal_acc_mean=('balanced_accuracy','mean'),
                bal_acc_std=('balanced_accuracy','std'),
                f1_mean=('macro_f1','mean'),
                recL23=('recall_L2/3','mean'),
                recL4=('recall_L4','mean'),
                recL5=('recall_L5','mean'),
                recL6=('recall_L6','mean'),
                n_held_out=('held_out_scan','count'))
           ).round(3).reset_index()
print('=== LOSO mean ± std across well-behaved scans ===')
print(summary.to_string(index=False))

## 10. Comparison panel

Side-by-side: within-session GKF on the random session vs LOSO on the
well-behaved pool. Plus baselines from notebook 07 §11d (full
13-scan dataset, no mismatch removal) and notebook 11 v1 / 08.

In [ ]:
# Build a unified comparison table.
rows_cmp = []
# A. Within-session (random session, clean).
for _, r in df_within.iterrows():
    rows_cmp.append({
        'protocol': f'within-session (clean, {WITHIN_SESSION})',
        'model': r['model'], 'preprocessing': r['preprocessing'], 'features': r['features'],
        'bal_acc': r['balanced_accuracy'], 'std': r['balanced_accuracy_std'],
        'macro_f1': r['macro_f1'],
    })
# B. LOSO on well-behaved pool (clean).
for _, r in summary.iterrows():
    rows_cmp.append({
        'protocol': f'LOSO (clean, well-behaved pool, n={r["n_held_out"]})',
        'model': r['model'], 'preprocessing': r['preprocessing'], 'features': r['features'],
        'bal_acc': r['bal_acc_mean'], 'std': r['bal_acc_std'],
        'macro_f1': r['f1_mean'],
    })
# C. Baselines from notebook 07 §11d (full dataset, no cleaning).
n07_paths = {
    ('HGB', 'raw',     'A1+B+C1+D1'):     'phase1_headline_loso_HGB_raw.parquet',
    ('HGB', 'zscored', 'A1+B+C1+D1'):     'phase1_headline_loso.parquet',
    ('DeepSets','raw','A1+B+C1+D1'):      'phase1_headline_loso_DeepSets_raw.parquet',
    ('DeepSets','zscored','A1+B+C1+D1'):  'phase1_headline_loso_DeepSets_zscored.parquet',
}
for (m, p, f), path in n07_paths.items():
    p_full = PROCESSED_RESULTS_DIR / path
    if not p_full.exists():
        continue
    df_b = pd.read_parquet(p_full)
    rows_cmp.append({
        'protocol': 'LOSO (full 13 scans, baseline)',
        'model': m, 'preprocessing': p, 'features': f,
        'bal_acc': df_b['balanced_accuracy'].mean(),
        'std': df_b['balanced_accuracy'].std(),
        'macro_f1': df_b['macro_f1'].mean(),
    })
# Tier G HGB LOSO baseline (from notebook 08).
tg_loso_path = PROCESSED_RESULTS_DIR / 'tierG_loso.parquet'
if tg_loso_path.exists():
    df_tg = pd.read_parquet(tg_loso_path)
    rows_cmp.append({
        'protocol': 'LOSO (full 13 scans, baseline)',
        'model': 'HGB', 'preprocessing': 'raw', 'features': 'TierG',
        'bal_acc': df_tg['balanced_accuracy'].mean(),
        'std': df_tg['balanced_accuracy'].std(),
        'macro_f1': df_tg['macro_f1'].mean(),
    })
df_cmp = pd.DataFrame(rows_cmp)
print('=== Unified comparison table ===')
print(df_cmp.round(3).to_string(index=False))

In [ ]:
# Plot — clean LOSO vs full LOSO baseline, per model/preprocessing.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Bar: clean LOSO vs full-dataset LOSO baseline, 5 model-norm combos.
ax = axes[0]
labels = ['HGB raw\nA1+B+C1+D1', 'HGB zs\nA1+B+C1+D1',
          'DS raw\nA1+B+C1+D1', 'DS zs\nA1+B+C1+D1',
          'HGB raw\nTier G']
x = np.arange(len(labels))
width = 0.35
clean_vals = []; clean_stds = []
full_vals  = []; full_stds  = []
for combo in [('HGB','raw','A1+B+C1+D1'), ('HGB','zscored','A1+B+C1+D1'),
              ('DeepSets','raw','A1+B+C1+D1'), ('DeepSets','zscored','A1+B+C1+D1'),
              ('HGB','raw','TierG')]:
    clean_row = df_cmp[(df_cmp['protocol'].str.startswith('LOSO (clean')) & (df_cmp['model']==combo[0])
                        & (df_cmp['preprocessing']==combo[1]) & (df_cmp['features']==combo[2])]
    full_row  = df_cmp[(df_cmp['protocol'].str.startswith('LOSO (full'))  & (df_cmp['model']==combo[0])
                        & (df_cmp['preprocessing']==combo[1]) & (df_cmp['features']==combo[2])]
    clean_vals.append(float(clean_row['bal_acc'].iloc[0]) if len(clean_row) else np.nan)
    clean_stds.append(float(clean_row['std'].iloc[0])     if len(clean_row) else np.nan)
    full_vals.append( float(full_row['bal_acc'].iloc[0])  if len(full_row)  else np.nan)
    full_stds.append( float(full_row['std'].iloc[0])      if len(full_row)  else np.nan)
ax.bar(x-width/2, full_vals,  yerr=full_stds,  width=width, label='LOSO full 13 scans (baseline)', color='gray')
ax.bar(x+width/2, clean_vals, yerr=clean_stds, width=width, label='LOSO clean well-behaved pool', color='steelblue')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.axhline(0.250, ls=':', color='red', alpha=0.5, label='chance')
ax.set_ylabel('balanced accuracy (LOSO)')
ax.set_title('Clean vs full-dataset LOSO')
ax.legend(loc='lower right', fontsize=9); ax.set_ylim(0.20, 0.75)

# (b) Within-session GKF on the random session.
ax = axes[1]
ws = df_within.copy()
labels_ws = [f'{r["model"]} {r["preprocessing"]}\n{r["features"]}' for _, r in ws.iterrows()]
ax.bar(np.arange(len(ws)), ws['balanced_accuracy'].values, yerr=ws['balanced_accuracy_std'].values, color='firebrick')
ax.set_xticks(np.arange(len(ws))); ax.set_xticklabels(labels_ws, fontsize=9, rotation=15)
ax.set_ylabel('balanced accuracy (within-session GKF)')
ax.set_title(f'Within-session GKF on {WITHIN_SESSION} (clean)')
ax.set_ylim(0.20, 0.85)
for i, v in enumerate(ws['balanced_accuracy'].values):
    ax.text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '11_clean_panel.png', dpi=130)
plt.show()

## 11. Interpretation

Both experiments completed. The clean-subset analysis produces three
important findings, two confirmatory and one *new* with implications for
the §11d (notebook 07) headline reconsideration.

### (a) Well-behaved pool: 5 of 13 scans

Mismatch removal + the (4 classes, min ≥ 20 per class) threshold gives
**5 well-behaved sessions**: `5_6, 5_7, 6_2, 6_4, 6_6`. 6_7 falls just
below threshold (L6 → 15 after cleaning, was 48 originally — most of
6_7's L6 cells were mismatched). The 7 single-class / 3-class sessions
were already excluded.

Random pick for within-session experiment: **5_6**, the cleanest
possible session (only 1 mismatch in the entire scan).

### (b) Within-session GKF on the random clean session — Tier G dominates

| model | prep | feats | bal_acc | macro_f1 | per-class recall (L2/3 / L4 / L5 / L6) |
|---|---|---|---|---|---|
| HGB | raw | long-row | 0.546 ± 0.017 | 0.480 | 0.14 / 0.56 / 0.49 / **0.99** |
| HGB | zscored | long-row | 0.545 ± 0.021 | 0.478 | 0.14 / 0.56 / 0.49 / 0.99 |
| DeepSets | raw | long-row | 0.561 ± 0.026 | 0.495 | 0.35 / 0.20 / **0.79** / 0.91 |
| DeepSets | zscored | long-row | 0.529 ± 0.020 | 0.471 | 0.29 / 0.34 / 0.55 / 0.94 |
| **HGB** | **raw** | **TierG** | **0.688 ± 0.046** | **0.687** | 0.51 / 0.71 / 0.63 / 0.90 |

**Tier G HGB is unambiguously the best within-session model** on clean
data (+0.127 over the second-best). It is also the only model with
balanced per-class recall — the long-row HGB completely collapses onto
L6 (recall 0.99 with L2/3 at 0.14). Clean within-session Tier G (0.688)
is +0.025 above the Phase-1 GKF Tier G (0.663) — the within-session
ceiling on clean data is slightly above the GKF ceiling on the full
13-scan dataset.

**Methodological confirmation**: zscoring within one session is
essentially neutral. HGB raw and zscored long-row give 0.546 vs 0.545 —
different by less than the fold std. Within a single session there are
no per-session offsets to remove, and HGB's tree splits are scale-
invariant. For Tier G specifically (all 116 features in `amp_mean`
units), within-session zscoring would be a no-op for HGB — confirming
the methodological note in §2.

### (c) LOSO on the clean well-behaved pool — DeepSets zscored takes the lead

| model | prep | feats | bal_acc | macro_f1 | per-class recall |
|---|---|---|---|---|---|
| **DeepSets** | **zscored** | long-row | **0.542 ± 0.035** | 0.419 | 0.13 / 0.55 / 0.63 / 0.87 |
| HGB | raw | TierG | 0.533 ± 0.050 | **0.510** | **0.64** / 0.36 / 0.50 / 0.63 |
| DeepSets | raw | long-row | 0.530 ± 0.033 | 0.433 | 0.26 / 0.39 / 0.65 / 0.82 |
| HGB | raw | long-row | 0.508 ± 0.014 | 0.375 | 0.21 / 0.51 / 0.36 / 0.96 |
| HGB | zscored | long-row | 0.482 ± 0.013 | 0.353 | 0.28 / 0.43 / 0.28 / 0.94 |

Three things to notice:

**(i) The §11d 'raw beats zscored under LOSO' pattern reverses on clean
data.** In the §11d full-13-scan analysis we found:

- DeepSets raw 0.520 > DeepSets zscored 0.485 (Δ = +0.035 favouring raw)
- HGB raw 0.445 > HGB zscored 0.347 (Δ = +0.098 favouring raw)

On clean data:

- **DeepSets zscored 0.542 > DeepSets raw 0.530** (Δ = +0.012 favouring zscored)
- HGB raw 0.508 > HGB zscored 0.482 (Δ = +0.026 favouring raw, but smaller gap)

**The §11d 'zscoring hurts LOSO' interpretation was data-quality
dependent, not structural.** The 7 odd scans (4 high-mismatch session-6,
plus 4_7 + 7_3 + 7_5 + 8_5 single-/three-class) were depressing the
zscored model under LOSO because per-session statistics computed on
those scans don't represent the underlying biology. On clean data,
zscoring is again the methodologically aligned choice — at minimum
neutral (DeepSets) or only slightly negative (HGB long-row), and
potentially beneficial.

**(ii) Per-class recall under LOSO clean is more nuanced.** Tier G HGB
is the only model with strong L2/3 recall (0.64); DeepSets variants
have L2/3 collapsing to 0.13–0.26 but recover L5 (0.63–0.65) much
better than HGB long-row (0.28–0.36). The class trade-off depends on
the model architecture: DeepSets uses set-pooling that integrates
many stimulus responses → benefits the rare-but-distinctive classes
(L5, L6); HGB long-row treats each row independently → benefits the
majority class L2/3 less when class weighting is balanced.

**(iii) Tier G HGB has the best macro_f1 (0.510)** under LOSO on clean
data. Despite DeepSets zscored having higher bal_acc (0.542 vs 0.533),
Tier G HGB has more balanced per-class recovery — the per-class range
is 0.36–0.64 (range 0.28) vs DeepSets zscored's 0.13–0.87 (range 0.74).
**Tier G HGB remains the most-balanced classifier across protocols.**

### (d) The cleaning effect — who benefits most

Compared to the full-13-scan baseline from notebook 07 §11d:

| candidate | clean LOSO | full-13 LOSO | Δ |
|---|---|---|---|
| HGB zscored long-row | 0.482 | 0.347 | **+0.135** (biggest lift) |
| HGB raw long-row | 0.508 | 0.445 | +0.063 |
| DeepSets zscored long-row | 0.542 | 0.485 | +0.057 |
| HGB Tier G | 0.533 | 0.497 | +0.036 |
| DeepSets raw long-row | 0.530 | 0.520 | +0.010 |

**HGB zscored long-row is the biggest beneficiary** (+0.135). Cleaning
the data recovers most of the LOSO penalty that the dirty-data §11d
analysis attributed to the 'zscoring hurts LOSO' phenomenon. The
phenomenon was real but it was driven by 4 high-mismatch session-6 scans
and the 7 odd scans, not by zscoring per se.

**DeepSets raw improves the least** (+0.010). DeepSets raw was already
robust to the dirty-data confounds because set-pooling averages out per-
row noise. Removing the noise barely helps — the model was already at
near its ceiling.

The methodological lesson: **the §11d 'DeepSets raw is the cross-
session champion' finding should be re-framed**. DeepSets raw is the
*most data-robust* model — it works similarly well on clean and dirty
data. But on clean data, DeepSets zscored slightly outperforms it,
and Tier G HGB matches both with better class balance.

### (e) Cross-experiment comparison — within-session vs LOSO under clean

Tier G HGB:

- Within-session GKF on clean 5_6: **0.688**
- LOSO on clean well-behaved pool: **0.533**
- Δ (within − LOSO): 0.155

This is similar to the Phase-1 Δ for Tier G (GKF 0.663 − LOSO 0.497 =
0.166). **The cross-session penalty for Tier G is essentially the
same on clean data as on the full dataset** — the cleaning improves
both endpoints by the same amount (~0.025-0.035). The penalty is
structural (limited training data per held-out scan, class-distribution
shift across scans), not a data-quality artifact.

DeepSets zscored long-row:

- Within-session GKF on clean 5_6: 0.529
- LOSO on clean well-behaved pool: 0.542
- Δ (within − LOSO): −0.013 (LOSO actually beats within-session)

This is interesting: DeepSets gets slightly *better* under LOSO than
within-session on the clean subset. Mechanism: DeepSets benefits from
more training data (LOSO sees ~4 sessions × ~700 = ~2,800 neurons vs
within-session 5_6 = 685 neurons). For a deep model, the cross-session
training data outweighs the cross-session distribution shift.

## 12. Conclusions

**Three substantive findings**:

1. **Tier G HGB remains the best within-session model** on clean data
   (0.688 within-session GKF on session 5_6, vs 0.561 DeepSets raw and
   0.546 HGB long-row). It also has the most balanced per-class recall
   (range 0.51–0.90). The Phase-1 within-protocol headline framing is
   reaffirmed.

2. **The §11d 'raw beats zscored under LOSO' phenomenon was data-
   quality dependent.** On the clean well-behaved subset:
   - DeepSets *zscored* (0.542) slightly beats DeepSets raw (0.530)
   - HGB zscored long-row recovers from 0.347 to 0.482 (Δ = +0.135)

   The §11d finding remains valid for the full-13-scan dataset, but
   the *interpretation* should be: zscoring is methodologically
   aligned and works fine on clean data; on dirty data (4 high-
   mismatch scans + 7 odd scans), zscoring's per-session statistics
   carry too much noise.

3. **The clean-subset cross-session ceiling is ~0.54** (DeepSets
   zscored 0.542, Tier G HGB 0.533, DeepSets raw 0.530). All three
   models cluster within 0.012. The Phase-1 LOSO numbers (full 13
   scans) sit ~0.03–0.05 below this ceiling — the difference is
   attributable to the data-quality issues identified in notebook 10
   and notebook 11 v1, not to fundamental model failure.

**What this notebook does NOT do.**

- It does NOT replace the Phase-1 reportable. The full-13-scan numbers
  remain the canonical reportables because they describe the full
  dataset performance.
- It does NOT establish a new headline candidate. DeepSets zscored
  beats Tier G HGB by 0.009 on clean LOSO bal_acc, but Tier G HGB has
  better macro_f1 (0.510 vs 0.419) and more balanced per-class recall.
  The two are complementary, not strictly ranked.

**Bridge to the report**: the clean-subset findings should be added
to `PHASE1_GROUND_TRUTH.md` as a new sub-section in §6 (Phase-1
characterization) documenting:

- The data-quality sensitivity: clean LOSO improves by +0.04–0.14
  depending on the model, with HGB zscored long-row being the biggest
  beneficiary.
- The §11d-pattern reversal: 'raw beats zscored under LOSO' was data-
  quality dependent and should be reframed in the report.
- The within-session ceiling under clean conditions (Tier G HGB =
  0.688 on 5_6) is +0.025 above the full-13-scan GKF ceiling.

**Bridge to Phase 2**: notebook 11 v2 confirms that **5_6 is a safe
starting session for the staged CNN analysis** described in
`PHASE2_PLANNING.md`. It has only 1 mismatch, balanced per-class
counts, and within-session Tier G HGB performance of 0.688 — meaning
any Phase-2 CNN that *works* on this session has reasonable signal to
exceed.

**Output artefacts**:

- `data/processed/results/clean_subset_per_session.parquet` — 13 rows.
- `data/processed/results/clean_within_session_runs.parquet` — 5 rows.
- `data/processed/results/clean_loso_runs.parquet` — 25 rows (5 model-
  norm combos × 5 held-out well-behaved scans).
- `reports/figures/11_clean_panel.png` — comparison figure.